In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import torch
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from tqdm import tqdm
import random
import os
import time
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# ================= 路径配置 =================
# 设置保存结果的根目录（与PINN程序相同）
RESULTS_ROOT = "./generated/mc_results"  

# 为蒙特卡洛模拟创建带时间戳的子文件夹
timestamp = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = f"run_mc_pi100_K50_{timestamp}"
SAVE_DIR = os.path.join(RESULTS_ROOT, RUN_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Monte Carlo results will be saved to: {SAVE_DIR}")

# ================= 参数设置 =================
pi = 100.0  # 参数 pi
beta = 0.5  # 参数 beta
alpha = 1.2  # 参数 alpha
r = 0.05    # 无风险利率
T_asset = 4.0  # 资产价格相关参数
tau = 3.0   # 期权到期时间
K = 50.0    # 行权价

# ================= 随机种子 =================
SEED = 100
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# ================= 蒙特卡洛模拟参数 =================
dt = 0.01           # 时间步长
N = int(tau / dt)   # 时间步数量
n_simulations = 10000  # 模拟路径数量

# 定义资产价格和时间网格
A_values = np.linspace(1, 120, 120)  # 避免0和100边界，防止数值问题
t_values = np.linspace(0, tau, 100)  # 时间点范围

# 存储蒙特卡洛价格
mc_prices = np.zeros((len(A_values), len(t_values)))

# ================= 蒙特卡洛模拟函数 =================
def monte_carlo_option_price(A0, t_start, n_simulations=5000):
    """
    计算给定初始资产价格A0和起始时间t_start的期权价格
    """
    # 如果起始时间接近到期时间，直接计算终端价值
    if tau - t_start < 1e-6:
        return np.maximum(A0 - K, 0)
    
    # 调整时间步
    steps = int((tau - t_start) / dt)
    if steps <= 0:
        return np.maximum(A0 - K, 0) * np.exp(-r * (tau - t_start))
    
    # 初始化路径
    A_paths = np.zeros((n_simulations, steps+1))
    A_paths[:, 0] = A0
    
    # 模拟路径
    for i in range(steps):
        t_current = t_start + i * dt
        time_diff = T_asset - t_current
        
        # 确保时间差为正
        if time_diff < 1e-6:
            time_diff = 1e-6
            
        sqrt_zt = np.sqrt(beta * time_diff**(-alpha))
        
        # 计算 Φ⁻¹(A/π) 和 Φ'(Φ⁻¹(A/π))
        A_ratio = A_paths[:, i] / pi
        A_ratio = np.clip(A_ratio, 1e-6, 1-1e-6)  # 避免边界值
        Phi_inv_A = stats.norm.ppf(A_ratio)
        Phi_prime = stats.norm.pdf(Phi_inv_A)
        
        # 布朗运动增量
        dW = np.sqrt(dt) * np.random.randn(n_simulations)
        dA = pi * Phi_prime * sqrt_zt * dW
        
        # 更新资产价格（避免超出边界）
        A_paths[:, i+1] = np.clip(A_paths[:, i] + dA, pi/10000, pi)
    
    # 计算期权价格
    payoffs = np.maximum(A_paths[:, -1] - K, 0)
    option_price = np.exp(-r * (tau - t_start)) * np.mean(payoffs)
    return option_price

# ================= 执行蒙特卡洛模拟 =================
print("进行蒙特卡洛模拟...")
start_time = time.time()

for i, A0 in enumerate(tqdm(A_values, desc="资产价格")):
    for j, t_start in enumerate(t_values):
        mc_prices[i, j] = monte_carlo_option_price(A0, t_start, n_simulations)

simulation_time = time.time() - start_time
print(f"蒙特卡洛模拟完成，用时 {simulation_time:.2f} 秒")

# 创建蒙特卡洛曲面数据
mc_A_grid, mc_t_grid = np.meshgrid(A_values, t_values, indexing='ij')
mc_A_flat = mc_A_grid.ravel()
mc_t_flat = mc_t_grid.ravel()
mc_prices_flat = mc_prices.ravel()

# 保存结果
mc_results = np.column_stack((mc_A_flat, mc_t_flat, mc_prices_flat))
results_path = os.path.join(SAVE_DIR, "mc_results.npy")
np.save(results_path, mc_results)

# ================= 绘图部分 =================
print("生成蒙特卡洛结果图...")

# 1. 3D曲面图
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_trisurf(mc_A_flat, mc_t_flat, mc_prices_flat, cmap='viridis', alpha=0.8)
ax.set_xlabel('Asset Price (A)')
ax.set_ylabel('Time (t)')
ax.set_zlabel('Option Price (C)')
ax.set_title(f'Monte Carlo Simulation: Carbon Option (K={K})')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Option Price')
plt.tight_layout()

surface_plot_path = os.path.join(SAVE_DIR, "mc_solution_3d.png")
plt.savefig(surface_plot_path, dpi=300)
plt.show()

# 2. 不同时间点的曲线图
time_points = np.linspace(0, tau, 6)
colors = cm.viridis(np.linspace(0, 1, len(time_points)))

plt.figure(figsize=(10, 6))

for t_val, color in zip(time_points, colors):
    # 找到最接近的时间点的索引
    t_idx = np.argmin(np.abs(t_values - t_val))
    
    # 提取该时间点的价格数据
    A_at_t = A_values
    C_at_t = mc_prices[:, t_idx]
    
    plt.plot(A_at_t, C_at_t, linewidth=2, label=f"t = {t_val:.2f}", color=color)

plt.xlabel("Asset Price (A)")
plt.ylabel("Option Price (C)")
plt.title(f"Monte Carlo: Carbon Option at Different Times (K={K})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

time_series_path = os.path.join(SAVE_DIR, "mc_time_series.png")
plt.savefig(time_series_path, dpi=300)
plt.show()

# 3. 2D等高线图
# 创建更精细的网格用于插值
A_range = np.linspace(min(A_values), max(A_values), 100)
t_range = np.linspace(min(t_values), max(t_values), 100)
A_mesh, t_mesh = np.meshgrid(A_range, t_range, indexing='ij')

# 插值蒙特卡洛结果
C_mc_grid = griddata(
    (mc_A_flat, mc_t_flat), 
    mc_prices_flat, 
    (A_mesh, t_mesh), 
    method='cubic'
)

plt.figure(figsize=(10, 7))
contour = plt.contourf(A_mesh, t_mesh, C_mc_grid, levels=50, cmap='viridis')
plt.contour(A_mesh, t_mesh, C_mc_grid, levels=10, colors='k', linewidths=0.5, alpha=0.5)

plt.colorbar(contour, label='Option Price (C)')
plt.xlabel('Asset Price (A)')
plt.ylabel('Time (t)')
plt.title(f'Monte Carlo: Carbon Option Price Surface (K={K})')

# 标记到期时间
plt.axhline(y=tau, color='r', linestyle='--', alpha=0.7, label='Expiration')
plt.legend()

plt.tight_layout()
contour_path = os.path.join(SAVE_DIR, "mc_price_surface_contour.png")
plt.savefig(contour_path, dpi=300)
plt.show()

# 4. 价格密度图（蒙特卡洛特有）
plt.figure(figsize=(10, 6))
sns.kdeplot(x=mc_A_flat, y=mc_prices_flat, cmap="viridis", fill=True, thresh=0.05)
plt.xlabel('Asset Price (A)')
plt.ylabel('Option Price (C)')
plt.title(f'Monte Carlo: Price Density (K={K})')
plt.grid(True, alpha=0.3)
plt.tight_layout()

density_plot_path = os.path.join(SAVE_DIR, "mc_price_density.png")
# plt.savefig(density_plot_path, dpi=300)
plt.show()

# ================= 保存参数信息 =================
params_info = f"""
Monte Carlo Simulation Parameters:
---------------------------------
Pi = {pi}
Beta = {beta}
Alpha = {alpha}
Risk-free rate (r) = {r}
T_asset = {T_asset}
Tau = {tau}
Strike price (K) = {K}

Simulation Parameters:
----------------------
Time steps (dt) = {dt}
Number of time steps (N) = {N}
Number of simulations = {n_simulations}
Asset price range = {A_values.min()} to {A_values.max()} (n={len(A_values)})
Time range = {t_values.min()} to {t_values.max()} (n={len(t_values)})
Simulation time = {simulation_time:.2f} seconds
"""

params_path = os.path.join(SAVE_DIR, "simulation_parameters.txt")
with open(params_path, "w") as f:
    f.write(params_info)

print(f"所有蒙特卡洛结果已保存至: {SAVE_DIR}")